# Nonogram environment demo

A nonogram is a logic puzzle where the numbers on the left and top describe consecutive blocks of filled cells. This notebook shows the clues, then applies actions one by one so the player grid visibly fills in.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML
from matplotlib import animation

from gym_nonogram.env import Nonogram, generate_count

In [ ]:
env = Nonogram(central_grid_size=5, seed=7, max_step=40)
state, info = env.reset(options={"central_grid_density": 0.45})

solution = env.solution_grid[env.small_grid_size :, env.small_grid_size :]
left_clues = env.solution_grid[env.small_grid_size :, : env.small_grid_size]
top_clues = env.solution_grid[: env.small_grid_size, env.small_grid_size :]

print("Hidden solution used for this demo:")
print(solution)
print("\nLeft clues:")
print(left_clues)
print("\nTop clues:")
print(top_clues)

In [ ]:
print("Row clue check:")
for index, row in enumerate(solution):
    print(index, generate_count(row))

print("\nColumn clue check:")
for index, col in enumerate(solution.T):
    print(index, generate_count(col))

In [ ]:
def plot_nonogram(env, state, title=""):
    small = env.small_grid_size
    player_grid = state[small:, small:]

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(np.zeros_like(state), cmap="gray", vmin=0, vmax=1)

    for row in range(state.shape[0]):
        for col in range(state.shape[1]):
            value = int(state[row, col])
            if row >= small and col >= small:
                mark = player_grid[row - small, col - small]
                color = "#222222" if mark == 2 else "#dddddd" if mark == 1 else "white"
                ax.add_patch(plt.Rectangle((col - 0.5, row - 0.5), 1, 1, facecolor=color, edgecolor="#999999"))
                if mark == 1:
                    ax.text(col, row, "x", ha="center", va="center", color="#555555", fontsize=12)
            elif value:
                ax.text(col, row, str(value), ha="center", va="center", color="#111111", fontsize=12)

    ax.axhline(small - 0.5, color="#111111", linewidth=2)
    ax.axvline(small - 0.5, color="#111111", linewidth=2)
    ax.set_xticks(np.arange(-0.5, state.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, state.shape[0], 1), minor=True)
    ax.grid(which="minor", color="#bbbbbb", linewidth=0.8)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)
    return fig, ax

plot_nonogram(env, state, "Initial puzzle: clues visible, player grid empty");

In [ ]:
actions = []
for row in range(env.central_grid_size):
    for col in range(env.central_grid_size):
        value = int(solution[row, col])
        actions.append((row, col, value))

frames = [state.copy()]
action_log = []
terminated = False
truncated = False

for step_index, action in enumerate(actions, start=1):
    state, reward, terminated, truncated, info = env.step(action)
    frames.append(state.copy())
    action_log.append((step_index, action, reward, terminated, truncated, info))
    if terminated or truncated:
        break

for item in action_log[:10]:
    print(item)
print(f"\nsteps={len(action_log)}, terminated={terminated}, truncated={truncated}")

In [ ]:
snapshot_indices = [0, 1, 5, 10, 15, len(frames) - 1]
snapshot_indices = sorted(set(index for index in snapshot_indices if index < len(frames)))

fig, axes = plt.subplots(1, len(snapshot_indices), figsize=(3 * len(snapshot_indices), 3))
if not isinstance(axes, np.ndarray):
    axes = np.array([axes])

for ax, frame_index in zip(axes, snapshot_indices):
    _, source_ax = plot_nonogram(env, frames[frame_index], f"Step {frame_index}")
    ax.imshow(source_ax.images[0].get_array(), cmap="gray")
    plt.close(source_ax.figure)

plt.close(fig)
for frame_index in snapshot_indices:
    plot_nonogram(env, frames[frame_index], f"Step {frame_index}");

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

def draw_frame(frame_index):
    ax.clear()
    frame = frames[frame_index]
    small = env.small_grid_size
    ax.imshow(np.zeros_like(frame), cmap="gray", vmin=0, vmax=1)
    for row in range(frame.shape[0]):
        for col in range(frame.shape[1]):
            value = int(frame[row, col])
            if row >= small and col >= small:
                color = "#222222" if value == 2 else "#dddddd" if value == 1 else "white"
                ax.add_patch(plt.Rectangle((col - 0.5, row - 0.5), 1, 1, facecolor=color, edgecolor="#999999"))
                if value == 1:
                    ax.text(col, row, "x", ha="center", va="center", color="#555555", fontsize=12)
            elif value:
                ax.text(col, row, str(value), ha="center", va="center", color="#111111", fontsize=12)
    ax.axhline(small - 0.5, color="#111111", linewidth=2)
    ax.axvline(small - 0.5, color="#111111", linewidth=2)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"Action {frame_index} / {len(frames) - 1}")

def update(frame_index):
    draw_frame(frame_index)

ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=450)
plt.close(fig)
HTML(ani.to_jshtml())